# **Pengembangan Model MLP: Arsitektur, Regularisasi, dan Training**

Notebook ini membangun arsitektur MLP untuk prediksi risiko diabetes dan menentukan konfigurasinya lewat eksperimen bertahap. Ada tiga hal yang dikerjakan:

1. Merancang arsitektur MLP beserta alasan teknis untuk jumlah layer, neuron, aktivasi, dropout, dan regularisasi L2.
2. Melatih model dengan optimizer Adam, EarlyStopping, dan ModelCheckpoint berdasarkan metrik validation.
3. Membaca kurva loss dan metrik train vs validation untuk melihat tanda overfitting atau underfitting.

Keputusan yang sudah dikunci kelompok sebelum eksperimen:

| Aspek | Keputusan |
|---|---|
| Aktivasi hidden layer | ReLU |
| Optimizer | Adam |
| Input pipeline | `tf.data` lewat helper `make_dataset()` |
| Pengulangan | Setiap konfigurasi dijalankan dengan 3 seed (42, 123, 2026) |
| Loss selama eksperimen | Binary cross-entropy (Focal Loss baru dipasang di notebook fitur unik 1) |
| Data yang dipakai | Train untuk melatih, validation untuk memilih. **Data test tidak disentuh** |

Learning rate dan batch size dituning di `04_eksperimen_optimasi.ipynb`. Di notebook ini keduanya ditahan di nilai awal.

**Penanggung jawab:** Wilbert Edric Tanjiro (231110595)

## **1. Import Library dan Konfigurasi**

Seluruh fungsi model, pipeline data, dan eksperimen ada di `src/model.py`, jadi notebook lain memakai kode yang sama persis.

Eksperimen di notebook ini melatih 30 model dan butuh waktu cukup lama (sekitar 30 sampai 60 menit di CPU, tergantung laptop). Karena itu ada saklar `RUN_EXPERIMENTS`:
- `True`: semua eksperimen dijalankan ulang, lalu hasilnya disimpan ke `reports/results/`.
- `False`: hasil eksperimen yang sudah tersimpan dimuat dari `reports/results/`, sehingga notebook selesai dalam hitungan detik.

In [1]:
import sys
import json
import time
import platform
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import keras

from src import config
from src import preprocessing as prep
from src import model as M

RUN_EXPERIMENTS = True

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})
WARNA_TRAIN, WARNA_VAL, WARNA_ABU, WARNA_PILIH = "#DD8452", "#4C72B0", "#9A9A9A", "#4C72B0"
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(config.FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("Python     :", platform.python_version())
print("Seed       :", M.SEEDS)
print("Konfigurasi awal:", M.DEFAULT_CONFIG)
print("RUN_EXPERIMENTS :", RUN_EXPERIMENTS)

TensorFlow : 2.21.0
Keras      : 3.15.1
Python     : 3.12.10
Seed       : [42, 123, 2026]
Konfigurasi awal: {'hidden_units': [128, 64], 'dropout': 0.2, 'l2': 0.0, 'batch_norm': False, 'learning_rate': 0.001, 'batch_size': 512}
RUN_EXPERIMENTS : True


## **2. Memuat Data Hasil Preprocessing**

Data diambil lewat `prepare_datasets()` dari `src/preprocessing.py`. Fingerprint data test dicocokkan dengan yang tersimpan di `models/preprocessing_config.json` untuk memastikan pembagian data sama dengan notebook 02. Data test ikut dimuat karena berasal dari fungsi yang sama, tetapi tidak dipakai sama sekali di notebook ini.

In [2]:
data = prep.prepare_datasets()
cfg_prep = prep.load_preprocessing_config()
assert data["test_fingerprint"] == cfg_prep["split"]["test_fingerprint"], "Split data berbeda dengan notebook 02"

for part in ["train", "val"]:
    X, y = data[f"X_{part}"], data[f"y_{part}"]
    print(f"{part:<5}: X {X.shape}, {X.dtype} | kelas 1 = {y.mean():.2%}")
print("Fingerprint test cocok:", data["test_fingerprint"])

train: X (160211, 21), float32 | kelas 1 = 15.30%
val  : X (34332, 21), float32 | kelas 1 = 15.30%
Fingerprint test cocok: 01b58633cb6f22b5


## **3. Pipeline Input dengan `tf.data`**

Data train dan validation diubah menjadi `tf.data.Dataset` lewat `M.make_dataset()` dengan urutan **shuffle, batch, prefetch**:
- **Shuffle** memakai buffer seukuran seluruh data train dengan seed tetap, dan diacak ulang di setiap epoch. Data validation tidak diacak.
- **Batch** membagi data menjadi kelompok kecil sesuai batch size.
- **Prefetch** menyiapkan batch berikutnya selagi model memproses batch saat ini.

Pengecekan di bawah memastikan pengacakannya bisa diulang. Dua dataset dengan seed sama harus menghasilkan batch pertama yang identik, sedangkan seed berbeda menghasilkan urutan yang berbeda.

In [3]:
bs = M.DEFAULT_CONFIG["batch_size"]
ds_a = M.make_dataset(data["X_train"], data["y_train"], bs, shuffle=True, seed=42)
ds_b = M.make_dataset(data["X_train"], data["y_train"], bs, shuffle=True, seed=42)
ds_c = M.make_dataset(data["X_train"], data["y_train"], bs, shuffle=True, seed=123)

xa, ya = next(iter(ds_a))
xb, _ = next(iter(ds_b))
xc, _ = next(iter(ds_c))

print("Element spec      :", ds_a.element_spec)
print("Jumlah batch train:", int(ds_a.cardinality()))
print("Bentuk satu batch :", tuple(xa.shape), tuple(ya.shape))
print("Seed sama  -> batch pertama identik :", bool(np.array_equal(xa.numpy(), xb.numpy())))
print("Seed beda  -> batch pertama identik :", bool(np.array_equal(xa.numpy(), xc.numpy())))
print(f"Proporsi kelas 1 di batch pertama  : {ya.numpy().mean():.2%}")

Element spec      : (TensorSpec(shape=(None, 21), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))
Jumlah batch train: 313
Bentuk satu batch : (512, 21) (512,)
Seed sama  -> batch pertama identik : True
Seed beda  -> batch pertama identik : False
Proporsi kelas 1 di batch pertama  : 17.97%


**Interpretasi:** Pipeline `tf.data` menghasilkan batch berukuran (512, 21) dengan tipe `float32`, dan satu epoch terdiri dari 313 batch. Angka 313 berasal dari 160.211 baris dibagi 512, dengan batch terakhir berisi sisa data yang lebih sedikit. Dua dataset dengan seed yang sama menghasilkan batch pertama yang identik, sedangkan seed berbeda menghasilkan urutan lain. Artinya pengacakan data sudah terkunci, dan ini salah satu syarat supaya hasil training bisa diulang.

Proporsi kelas 1 di batch pertama adalah 17,97%, sedikit di atas proporsi keseluruhan (15,30%). Variasi seperti ini wajar karena batch diambil acak. Dengan batch size 512, setiap batch tetap berisi puluhan sampel diabetes, jadi gradien dari kelas positif tidak hilang di satu batch tertentu.

## **4. Rancangan Arsitektur dan Alasan Teknis**

Arsitektur dirancang sebagai MLP berbentuk corong, yaitu jumlah neuron mengecil di setiap layer berikutnya. Setiap hidden layer tersusun dari **Dense, ReLU, lalu Dropout**. Kalau BatchNorm dipakai, posisinya di antara Dense dan ReLU.

| Komponen | Pilihan awal | Alasan teknis |
|---|---|---|
| Input | 21 neuron | Sesuai jumlah fitur setelah preprocessing |
| Jumlah hidden layer | 2 (diuji 2 dan 3) | Fitur sudah berupa indikator bermakna, jadi tidak butuh jaringan yang sangat dalam. Satu layer saja kurang leluasa memodelkan interaksi antarfitur, sedangkan 4 layer ke atas berisiko overfit tanpa manfaat jelas pada 21 fitur |
| Neuron | 128 lalu 64 (diuji 64 sampai 256 di layer pertama) | Layer pertama sekitar 3 sampai 12 kali jumlah input supaya cukup ruang untuk kombinasi fitur, lalu dibagi dua di layer berikutnya untuk merangkum informasi menuju output |
| Aktivasi | ReLU dengan inisialisasi He normal | ReLU murah dihitung dan tidak mengalami *vanishing gradient* seperti sigmoid atau tanh di hidden layer. Inisialisasi He dirancang khusus untuk ReLU (He dkk., 2015) |
| Dropout | 0,2 (diuji 0 sampai 0,3) | Mematikan sebagian neuron secara acak saat training supaya model tidak bergantung pada pola tertentu (Srivastava dkk., 2014) |
| Regularisasi L2 | Tidak ada (diuji 1e-5 dan 1e-4) | Menghukum bobot yang terlalu besar. Diuji terpisah karena EarlyStopping dan dropout mungkin sudah cukup |
| BatchNorm | Tidak ada (diuji) | Menstabilkan distribusi aktivasi (Ioffe & Szegedy, 2015). Input sudah di-scaling, jadi manfaatnya belum tentu terasa pada jaringan dangkal |
| Output | 1 neuron sigmoid | Menghasilkan probabilitas diabetes antara 0 dan 1 |
| Loss | Binary cross-entropy | Loss standar untuk klasifikasi biner dengan output probabilitas |
| Optimizer | Adam, learning rate 0,001 | Adam menyesuaikan learning rate per parameter dan stabil tanpa banyak tuning. Nilai 0,001 adalah nilai yang direkomendasikan di paper aslinya (Kingma & Ba, 2015) dan dituning lebih lanjut di notebook 04 |
| Batch size | 512 (sementara) | Cukup besar supaya setiap batch rata-rata berisi sekitar 78 sampel kelas 1, sehingga gradien tidak terlalu berisik. Dituning di notebook 04 |
| EarlyStopping | Pantau PR-AUC validation, patience 8, kembalikan bobot terbaik | Menghentikan training saat performa validation berhenti membaik (Prechelt, 1998) |
| ModelCheckpoint | Simpan bobot dengan PR-AUC validation terbaik | Bobot terbaik tersimpan ke file walaupun training masih berlanjut beberapa epoch setelahnya |
| Epoch maksimum | 100 | Batas atas saja, karena training hampir selalu berhenti lebih dulu oleh EarlyStopping |

**Alasan memakai PR-AUC, bukan accuracy.** Model yang selalu menebak "tidak diabetes" sudah mendapat accuracy sekitar 85%. PR-AUC (*average precision*) hanya berfokus pada seberapa baik model menemukan kelas positif, sehingga lebih informatif untuk data tidak seimbang (Saito & Rehmsmeier, 2015). Nilai acuannya adalah proporsi kelas positif, yaitu sekitar 0,153. Model yang tidak belajar apa pun akan mendapat PR-AUC di sekitar angka itu. ROC-AUC tetap dicatat sebagai metrik pendamping.

In [5]:
model_awal = M.build_model(**M.DEFAULT_CONFIG)
model_awal.summary()
print(f"Jumlah parameter yang dilatih: {model_awal.count_params():,}")

Model: "mlp_diabetes"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fitur_input (InputLayer)        │ (None, 21)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         2,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_1 (Activation)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_2 (Activation)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,137 (43.50 KB)

 Trainable params: 11,137 (43.50 KB)

 Non-trainable params: 0 (0.00 B)

Jumlah parameter yang dilatih: 11,137


**Interpretasi:** Model awal punya 11.137 parameter yang dilatih. Rinciannya: `dense_1` sebanyak 2.816 (21 x 128 bobot ditambah 128 bias), `dense_2` sebanyak 8.256 (128 x 64 ditambah 64), dan layer output sebanyak 65. Layer ReLU dan Dropout tidak punya parameter karena hanya mengubah nilai aktivasi.

Dibanding jumlah data train yang mencapai 160.211 baris, model ini tergolong kecil, yaitu sekitar 14 sampel untuk setiap parameter. Dengan perbandingan seperti ini, risiko model menghafal data train relatif kecil. Itu juga alasan kandidat terbesar di eksperimen E1 dibatasi sekitar 47 ribu parameter.

## **5. Protokol Eksperimen**

Eksperimen dilakukan bertahap. Setiap tahap mengubah satu kelompok faktor dan mengunci pemenang tahap sebelumnya:

| Tahap | Yang diuji | Kandidat |
|---|---|---|
| E1 | Kedalaman dan lebar | [64-32], [128-64], [256-128], [128-64-32], [256-128-64] |
| E2a | Dropout | 0, 0,2, 0,3 |
| E2b | Regularisasi tambahan | tanpa tambahan, L2 1e-5, L2 1e-4, BatchNorm |

**Aturan pemilihan** (fungsi `M.select_best()`):
1. Cari skenario dengan rata-rata PR-AUC validation tertinggi dari 3 seed.
2. Skenario lain dianggap **setara** kalau rata-ratanya tidak lebih rendah dari (terbaik dikurangi 1 standar deviasi skenario terbaik).
3. Dari skenario yang setara, pilih yang **paling sederhana**, yaitu komponen tambahan (L2, BatchNorm) paling sedikit, lalu jumlah parameter paling sedikit.

Aturan ini dipakai supaya selisih yang hanya terjadi karena kebetulan seed tidak dianggap sebagai keunggulan. Dengan 3 seed, standar deviasi masih berupa perkiraan kasar, jadi aturan ini sifatnya heuristik yang konservatif.

Konfigurasi yang persis sama dengan run sebelumnya tidak dilatih ulang, tetapi hasilnya dipakai lagi. Karena training sudah deterministik, hasilnya akan sama persis.

In [6]:
RUNS_PATH = config.RESULTS_DIR / "03_eksperimen_runs.csv"
HIST_PATH = config.RESULTS_DIR / "03_riwayat_training.json"

if RUN_EXPERIMENTS:
    cache, kumpulan_runs, kumpulan_hist = {}, [], {}
    waktu_mulai = time.time()
else:
    runs_tersimpan = pd.read_csv(RUNS_PATH)
    hist_tersimpan = M.load_json(HIST_PATH)
    print(f"Memuat {len(runs_tersimpan)} run dari {RUNS_PATH.relative_to(ROOT)}")

def jalankan_tahap(tahap, configs):
    """Menjalankan (atau memuat) semua skenario dalam satu tahap eksperimen."""
    if RUN_EXPERIMENTS:
        runs, hist = M.run_configs(configs, data, prefix=tahap, cache=cache)
        runs.insert(0, "tahap", tahap)
        kumpulan_runs.append(runs)
        kumpulan_hist.update({f"{tahap}|{k}": v for k, v in hist.items()})
    else:
        runs = runs_tersimpan[runs_tersimpan["tahap"] == tahap].reset_index(drop=True)
        assert list(dict.fromkeys(runs["skenario"])) == list(configs), "Skenario tersimpan tidak cocok"
        hist = {k.split("|", 1)[1]: v for k, v in hist_tersimpan.items() if k.startswith(f"{tahap}|")}
    return runs, hist

def tabel_ringkas(summary, terpilih):
    t = summary[["config", "params", "val_pr_auc_mean", "val_pr_auc_std", "val_roc_auc_mean",
                 "train_pr_auc_mean", "gap_pr_auc_mean", "best_epoch_mean", "seconds_mean",
                 "setara_dengan_terbaik"]].copy()
    t.columns = ["konfigurasi", "parameter", "val PR-AUC", "std", "val ROC-AUC", "train PR-AUC",
                 "gap train-val", "epoch terbaik", "detik/run", "setara terbaik"]
    t["dipilih"] = ["ya" if i == terpilih else "" for i in t.index]
    return t.round({"val PR-AUC": 4, "std": 4, "val ROC-AUC": 4, "train PR-AUC": 4,
                    "gap train-val": 4, "epoch terbaik": 1, "detik/run": 0})

def plot_tahap(runs, summary, terpilih, judul, nama_file):
    """Dot plot: titik kecil = tiap seed, titik besar + garis = rata-rata dan +-1 std."""
    urutan = list(summary.index)
    fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(urutan)), 4))
    for i, sk in enumerate(urutan):
        warna = WARNA_PILIH if sk == terpilih else WARNA_ABU
        nilai = runs.loc[runs["skenario"] == sk, "val_pr_auc"]
        ax.scatter(np.full(len(nilai), i) + np.array([-0.08, 0, 0.08])[:len(nilai)], nilai,
                   s=18, color=warna, alpha=0.5, zorder=2)
        m, s = summary.loc[sk, "val_pr_auc_mean"], summary.loc[sk, "val_pr_auc_std"]
        ax.errorbar(i + 0.25, m, yerr=s, fmt="o", ms=8, color=warna, capsize=4, lw=2, zorder=3)
        ax.annotate(f"{m:.4f}", (i + 0.33, m), va="center", fontsize=9, color="#333333")
    batas = summary.loc[summary["val_pr_auc_mean"].idxmax(), "val_pr_auc_mean"] - \
            summary.loc[summary["val_pr_auc_mean"].idxmax(), "val_pr_auc_std"]
    ax.axhline(batas, ls=":", color="#555555", lw=1)
    ax.text(-0.45, batas, "batas setara", va="bottom", ha="left", fontsize=8, color="#555555")
    ax.set_xticks(range(len(urutan)), [f"{sk}\n({summary.loc[sk, 'params']:,} param)" for sk in urutan])
    ax.set_xlim(-0.5, len(urutan) - 0.2)
    ax.set_ylabel("PR-AUC validation")
    ax.set_title(judul, fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color=WARNA_PILIH, label="dipilih"),
                       mpatches.Patch(color=WARNA_ABU, label="kandidat lain")],
              loc="lower right", fontsize=8)
    fig.text(0.01, -0.02, "Titik kecil = hasil tiap seed. Titik besar dan garis = rata-rata +- 1 std. "
             "Sumbu y diperbesar dan tidak dimulai dari 0.", fontsize=8, color="#555555")
    fig.tight_layout()
    save_fig(fig, nama_file)
    plt.show()